# UCI Online Retail RFM segmentation

This notebook trains the final model only from the official UCI Online Retail dataset (dataset 352, exactly 541,909 transaction rows). Its first code cell installs an embedded snapshot of the project's Python training package into the runtime temporary directory, so uploading only this notebook to Google Colab is sufficient. Colab then attempts the official UCI download first; if that fails, it offers manual upload of `Online Retail.xlsx`. Both paths apply the same checks before training. Metrics are computed at runtime; no synthetic data is used for the final model.

In [1]:
import base64
import sys
import tempfile
from pathlib import Path

# Bundled project modules make this notebook self-contained in Google Colab.
# Keep the embedded snapshots synchronized with src/segmenta_rfm_atlas/.
embedded_sources = {
    '__init__.py': (
        'IiIiUkZNIHNlZ21lbnRhdGlvbiBhcHBsaWNhdGlvbi4iIiIKCl9fdmVyc2lvbl9fID0gIjAuMS4wIgo='
    ),
    'dataset.py': (
        'IiIiQWNxdWlzaXRpb24gYW5kIHZhbGlkYXRpb24gb2YgdGhlIG9mZmljaWFsIFVDSSBPbmxpbmUgUmV0YWlsIHdvcmtib29rLiIi'
        'IgoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmltcG9ydCBoYXNobGliCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0'
        'aAppbXBvcnQgc2h1dGlsCmltcG9ydCB0ZW1wZmlsZQpmcm9tIHR5cGluZyBpbXBvcnQgQW55CmltcG9ydCB1cmxsaWIuZXJyb3IK'
        'aW1wb3J0IHVybGxpYi5yZXF1ZXN0CmltcG9ydCB6aXBmaWxlCgppbXBvcnQgcGFuZGFzIGFzIHBkCgoKY2xhc3MgRGF0YXNldExv'
        'YWRFcnJvcihSdW50aW1lRXJyb3IpOgogICAgIiIiUmFpc2VkIHdoZW4gdGhlIG9mZmljaWFsIHdvcmtib29rIGNhbm5vdCBiZSBk'
        'b3dubG9hZGVkIG9yIHZhbGlkYXRlZC4iIiIKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBEYXRhc2V0TG9hZFJlc3Vs'
        'dDoKICAgIHRyYW5zYWN0aW9uczogcGQuRGF0YUZyYW1lCiAgICBtZXRhZGF0YTogZGljdFtzdHIsIEFueV0KCgpjbGFzcyBPbmxp'
        'bmVSZXRhaWxEYXRhc2V0TG9hZGVyOgogICAgIiIiRG93bmxvYWQgb3IgbG9hZCBVQ0kgZGF0YXNldCAzNTIgYW5kIHZhbGlkYXRl'
        'IGl0cyB0cmFuc2FjdGlvbiBjb3VudC4iIiIKCiAgICBPRkZJQ0lBTF9EQVRBU0VUX0lEID0gMzUyCiAgICBPRkZJQ0lBTF9ET1dO'
        'TE9BRF9VUkwgPSAiaHR0cHM6Ly9hcmNoaXZlLmljcy51Y2kuZWR1L3N0YXRpYy9wdWJsaWMvMzUyL29ubGluZStyZXRhaWwuemlw'
        'IgogICAgRVhQRUNURURfVFJBTlNBQ1RJT05fQ09VTlQgPSA1NDFfOTA5CiAgICBSRVFVSVJFRF9DT0xVTU5TID0gKCJJbnZvaWNl'
        'Tm8iLCAiSW52b2ljZURhdGUiLCAiUXVhbnRpdHkiLCAiVW5pdFByaWNlIiwgIkN1c3RvbWVySUQiKQoKICAgIGRlZiBfX2luaXRf'
        'XyhzZWxmLCB0aW1lb3V0X3NlY29uZHM6IGludCA9IDkwLCBleHBlY3RlZF90cmFuc2FjdGlvbl9jb3VudDogaW50IHwgTm9uZSA9'
        'IE5vbmUpOgogICAgICAgIHNlbGYudGltZW91dF9zZWNvbmRzID0gdGltZW91dF9zZWNvbmRzCiAgICAgICAgc2VsZi5leHBlY3Rl'
        'ZF90cmFuc2FjdGlvbl9jb3VudCA9ICgKICAgICAgICAgICAgZXhwZWN0ZWRfdHJhbnNhY3Rpb25fY291bnQKICAgICAgICAgICAg'
        'aWYgZXhwZWN0ZWRfdHJhbnNhY3Rpb25fY291bnQgaXMgbm90IE5vbmUKICAgICAgICAgICAgZWxzZSBzZWxmLkVYUEVDVEVEX1RS'
        'QU5TQUNUSU9OX0NPVU5UCiAgICAgICAgKQoKICAgIGRlZiBsb2FkKHNlbGYsIHdvcmtib29rX3BhdGg6IFBhdGgsIGF1dG9fZG93'
        'bmxvYWQ6IGJvb2wgPSBUcnVlKSAtPiBEYXRhc2V0TG9hZFJlc3VsdDoKICAgICAgICAiIiJMb2FkIHRoZSBjb25maWd1cmVkIHdv'
        'cmtib29rLCBvcHRpb25hbGx5IGRvd25sb2FkaW5nIGl0IGZyb20gVUNJLiIiIgogICAgICAgIHdvcmtib29rX3BhdGggPSBQYXRo'
        'KHdvcmtib29rX3BhdGgpCiAgICAgICAgZG93bmxvYWRlZCA9IEZhbHNlCiAgICAgICAgaWYgbm90IHdvcmtib29rX3BhdGguaXNf'
        'ZmlsZSgpOgogICAgICAgICAgICBpZiBub3QgYXV0b19kb3dubG9hZDoKICAgICAgICAgICAgICAgIHJhaXNlIERhdGFzZXRMb2Fk'
        'RXJyb3IoCiAgICAgICAgICAgICAgICAgICAgZiJXb3JrYm9vayBub3QgZm91bmQgYXQge3dvcmtib29rX3BhdGh9LiBVcGxvYWQg'
        'dGhlIG9mZmljaWFsICIKICAgICAgICAgICAgICAgICAgICAiT25saW5lIFJldGFpbC54bHN4IHdvcmtib29rIGFuZCByZXRyeS4i'
        'CiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIHNlbGYuZG93bmxvYWRfZnJvbV91Y2kod29ya2Jvb2tfcGF0aCkKICAgICAg'
        'ICAgICAgZG93bmxvYWRlZCA9IFRydWUKICAgICAgICB0cnk6CiAgICAgICAgICAgIHRyYW5zYWN0aW9ucyA9IHBkLnJlYWRfZXhj'
        'ZWwod29ya2Jvb2tfcGF0aCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGVycm9yOgogICAgICAgICAgICByYWlzZSBEYXRh'
        'c2V0TG9hZEVycm9yKGYiQ291bGQgbm90IHJlYWQgT25saW5lIFJldGFpbCB3b3JrYm9vazoge2Vycm9yfSIpIGZyb20gZXJyb3IK'
        'ICAgICAgICBtaXNzaW5nID0gW2NvbHVtbiBmb3IgY29sdW1uIGluIHNlbGYuUkVRVUlSRURfQ09MVU1OUyBpZiBjb2x1bW4gbm90'
        'IGluIHRyYW5zYWN0aW9ucy5jb2x1bW5zXQogICAgICAgIGlmIG1pc3Npbmc6CiAgICAgICAgICAgIHJhaXNlIERhdGFzZXRMb2Fk'
        'RXJyb3IoZiJXb3JrYm9vayBpcyBtaXNzaW5nIHJlcXVpcmVkIFVDSSBjb2x1bW5zOiB7JywgJy5qb2luKG1pc3NpbmcpfSIpCiAg'
        'ICAgICAgYWN0dWFsX2NvdW50ID0gbGVuKHRyYW5zYWN0aW9ucykKICAgICAgICBpZiBhY3R1YWxfY291bnQgIT0gc2VsZi5leHBl'
        'Y3RlZF90cmFuc2FjdGlvbl9jb3VudDoKICAgICAgICAgICAgcmFpc2UgRGF0YXNldExvYWRFcnJvcigKICAgICAgICAgICAgICAg'
        'ICJSZWplY3RlZCBPbmxpbmUgUmV0YWlsIHdvcmtib29rOiBleHBlY3RlZCBleGFjdGx5ICIKICAgICAgICAgICAgICAgIGYie3Nl'
        'bGYuZXhwZWN0ZWRfdHJhbnNhY3Rpb25fY291bnQ6LH0gdHJhbnNhY3Rpb25zLCBmb3VuZCB7YWN0dWFsX2NvdW50Oix9LiIKICAg'
        'ICAgICAgICAgKQogICAgICAgIG1ldGFkYXRhID0gewogICAgICAgICAgICAiZGF0YXNldF9pZCI6IHNlbGYuT0ZGSUNJQUxfREFU'
        'QVNFVF9JRCwKICAgICAgICAgICAgImRhdGFzZXRfbmFtZSI6ICJPbmxpbmUgUmV0YWlsIiwKICAgICAgICAgICAgInNvdXJjZSI6'
        'ICJ1Y2ktb2ZmaWNpYWwtZG93bmxvYWQiIGlmIGRvd25sb2FkZWQgZWxzZSAibG9jYWwtdWNpLXdvcmtib29rIiwKICAgICAgICAg'
        'ICAgInNvdXJjZV91cmwiOiBzZWxmLk9GRklDSUFMX0RPV05MT0FEX1VSTCwKICAgICAgICAgICAgInRyYW5zYWN0aW9uX2NvdW50'
        'IjogYWN0dWFsX2NvdW50LAogICAgICAgICAgICAic2hhMjU2Ijogc2VsZi5fc2hhMjU2KHdvcmtib29rX3BhdGgpLAogICAgICAg'
        'IH0KICAgICAgICByZXR1cm4gRGF0YXNldExvYWRSZXN1bHQodHJhbnNhY3Rpb25zPXRyYW5zYWN0aW9ucywgbWV0YWRhdGE9bWV0'
        'YWRhdGEpCgogICAgZGVmIGRvd25sb2FkX2Zyb21fdWNpKHNlbGYsIHdvcmtib29rX3BhdGg6IFBhdGgpIC0+IFBhdGg6CiAgICAg'
        'ICAgIiIiRmV0Y2ggdGhlIG9mZmljaWFsIFVDSSBaSVAgYW5kIGF0b21pY2FsbHkgZXh0cmFjdCBpdHMgd29ya2Jvb2suIiIiCiAg'
        'ICAgICAgd29ya2Jvb2tfcGF0aCA9IFBhdGgod29ya2Jvb2tfcGF0aCkKICAgICAgICB3b3JrYm9va19wYXRoLnBhcmVudC5ta2Rp'
        'cihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgZXh0cmFjdGVkX3BhdGggPSB3b3JrYm9va19wYXRoLndpdGhf'
        'bmFtZSh3b3JrYm9va19wYXRoLm5hbWUgKyAiLnBhcnQiKQogICAgICAgIHRyeToKICAgICAgICAgICAgd2l0aCB0ZW1wZmlsZS5U'
        'ZW1wb3JhcnlEaXJlY3RvcnkoZGlyPXdvcmtib29rX3BhdGgucGFyZW50KSBhcyB0ZW1wb3JhcnlfZGlyZWN0b3J5OgogICAgICAg'
        'ICAgICAgICAgYXJjaGl2ZV9wYXRoID0gUGF0aCh0ZW1wb3JhcnlfZGlyZWN0b3J5KSAvICJ1Y2ktb25saW5lLXJldGFpbC56aXAi'
        'CiAgICAgICAgICAgICAgICByZXF1ZXN0ID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCgKICAgICAgICAgICAgICAgICAgICBzZWxm'
        'Lk9GRklDSUFMX0RPV05MT0FEX1VSTCwKICAgICAgICAgICAgICAgICAgICBoZWFkZXJzPXsiVXNlci1BZ2VudCI6ICJTZWdtZW50'
        'YS1SRk0tQXRsYXMvMC4xIGRhdGFzZXQgbG9hZGVyIn0sCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICB3aXRoIHVy'
        'bGxpYi5yZXF1ZXN0LnVybG9wZW4ocmVxdWVzdCwgdGltZW91dD1zZWxmLnRpbWVvdXRfc2Vjb25kcykgYXMgcmVzcG9uc2U6CiAg'
        'ICAgICAgICAgICAgICAgICAgd2l0aCBhcmNoaXZlX3BhdGgub3Blbigid2IiKSBhcyBhcmNoaXZlX2ZpbGU6CiAgICAgICAgICAg'
        'ICAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihyZXNwb25zZSwgYXJjaGl2ZV9maWxlKQogICAgICAgICAgICAgICAgd2l0'
        'aCB6aXBmaWxlLlppcEZpbGUoYXJjaGl2ZV9wYXRoKSBhcyBhcmNoaXZlOgogICAgICAgICAgICAgICAgICAgIHdvcmtib29rX21l'
        'bWJlciA9IG5leHQoCiAgICAgICAgICAgICAgICAgICAgICAgICgKICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1lbWJlcgog'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIG1lbWJlciBpbiBhcmNoaXZlLm5hbWVsaXN0KCkKICAgICAgICAgICAgICAg'
        'ICAgICAgICAgICAgIGlmIFBhdGgobWVtYmVyKS5uYW1lLmNhc2Vmb2xkKCkgPT0gIm9ubGluZSByZXRhaWwueGxzeCIKICAgICAg'
        'ICAgICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgICAgICAgICAgICAgTm9uZSwKICAgICAgICAgICAgICAgICAgICApCiAg'
        'ICAgICAgICAgICAgICAgICAgaWYgd29ya2Jvb2tfbWVtYmVyIGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNl'
        'IERhdGFzZXRMb2FkRXJyb3IoIlRoZSBvZmZpY2lhbCBVQ0kgYXJjaGl2ZSBkaWQgbm90IGNvbnRhaW4gT25saW5lIFJldGFpbC54'
        'bHN4LiIpCiAgICAgICAgICAgICAgICAgICAgd2l0aCBhcmNoaXZlLm9wZW4od29ya2Jvb2tfbWVtYmVyKSBhcyBzb3VyY2UsIGV4'
        'dHJhY3RlZF9wYXRoLm9wZW4oIndiIikgYXMgdGFyZ2V0OgogICAgICAgICAgICAgICAgICAgICAgICBzaHV0aWwuY29weWZpbGVv'
        'Ymooc291cmNlLCB0YXJnZXQpCiAgICAgICAgICAgIGV4dHJhY3RlZF9wYXRoLnJlcGxhY2Uod29ya2Jvb2tfcGF0aCkKICAgICAg'
        'ICBleGNlcHQgKHVybGxpYi5lcnJvci5VUkxFcnJvciwgVGltZW91dEVycm9yLCB6aXBmaWxlLkJhZFppcEZpbGUsIE9TRXJyb3Is'
        'IFN0b3BJdGVyYXRpb24pIGFzIGVycm9yOgogICAgICAgICAgICBleHRyYWN0ZWRfcGF0aC51bmxpbmsobWlzc2luZ19vaz1UcnVl'
        'KQogICAgICAgICAgICByYWlzZSBEYXRhc2V0TG9hZEVycm9yKGYiQ291bGQgbm90IGRvd25sb2FkL2V4dHJhY3QgdGhlIG9mZmlj'
        'aWFsIFVDSSB3b3JrYm9vazoge2Vycm9yfSIpIGZyb20gZXJyb3IKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAg'
        'ICBleHRyYWN0ZWRfcGF0aC51bmxpbmsobWlzc2luZ19vaz1UcnVlKQogICAgICAgICAgICByYWlzZQogICAgICAgIHJldHVybiB3'
        'b3JrYm9va19wYXRoCgogICAgZGVmIHVwbG9hZF9tYW51YWxfd29ya2Jvb2soc2VsZiwgd29ya2Jvb2tfcGF0aDogUGF0aCkgLT4g'
        'UGF0aDoKICAgICAgICAiIiJJbiBDb2xhYiwgdXBsb2FkIHRoZSBmYWxsYmFjayB3b3JrYm9vayBhbmQgc2F2ZSBpdCBhdCB3b3Jr'
        'Ym9va19wYXRoLiIiIgogICAgICAgIGlmIG5vdCBzZWxmLmlzX2dvb2dsZV9jb2xhYigpOgogICAgICAgICAgICByYWlzZSBEYXRh'
        'c2V0TG9hZEVycm9yKCJNYW51YWwgYnJvd3NlciB1cGxvYWQgaXMgYXZhaWxhYmxlIG9ubHkgaW4gR29vZ2xlIENvbGFiLiIpCiAg'
        'ICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IGZpbGVzCgogICAgICAgIHVwbG9hZGVkID0gZmlsZXMudXBsb2FkKCkKICAg'
        'ICAgICBtYXRjaGluZyA9IFsKICAgICAgICAgICAgKGZpbGVuYW1lLCBjb250ZW50KQogICAgICAgICAgICBmb3IgZmlsZW5hbWUs'
        'IGNvbnRlbnQgaW4gdXBsb2FkZWQuaXRlbXMoKQogICAgICAgICAgICBpZiBQYXRoKGZpbGVuYW1lKS5uYW1lLmNhc2Vmb2xkKCkg'
        'PT0gIm9ubGluZSByZXRhaWwueGxzeCIKICAgICAgICBdCiAgICAgICAgaWYgbm90IG1hdGNoaW5nOgogICAgICAgICAgICByYWlz'
        'ZSBEYXRhc2V0TG9hZEVycm9yKCJVcGxvYWQgYSBmaWxlIG5hbWVkIGV4YWN0bHkgT25saW5lIFJldGFpbC54bHN4LiIpCiAgICAg'
        'ICAgd29ya2Jvb2tfcGF0aCA9IFBhdGgod29ya2Jvb2tfcGF0aCkKICAgICAgICB3b3JrYm9va19wYXRoLnBhcmVudC5ta2Rpcihw'
        'YXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgd29ya2Jvb2tfcGF0aC53cml0ZV9ieXRlcyhtYXRjaGluZ1swXVsx'
        'XSkKICAgICAgICByZXR1cm4gd29ya2Jvb2tfcGF0aAoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBpc19nb29nbGVfY29sYWIo'
        'KSAtPiBib29sOgogICAgICAgICIiIlJldHVybiB3aGV0aGVyIHRoaXMgUHl0aG9uIHByb2Nlc3MgaXMgcnVubmluZyBpbiBHb29n'
        'bGUgQ29sYWIuIiIiCiAgICAgICAgaW1wb3J0IG9zCgogICAgICAgIHJldHVybiAiQ09MQUJfUkVMRUFTRV9UQUciIGluIG9zLmVu'
        'dmlyb24KCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX3NoYTI1NihwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICAgICAgZGlnZXN0'
        'ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCJyYiIpIGFzIGZpbGVfaGFuZGxlOgogICAg'
        'ICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1iZGE6IGZpbGVfaGFuZGxlLnJlYWQoMTAyNCAqIDEwMjQpLCBiIiIpOgogICAg'
        'ICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShibG9jaykKICAgICAgICByZXR1cm4gZGlnZXN0LmhleGRpZ2VzdCgpCg=='
    ),
    'modeling.py': (
        'IiIiQ2xhc3MtYmFzZWQgUkZNIHByZXBhcmF0aW9uLCB0cmFpbmluZywgZXZhbHVhdGlvbiwgYW5kIGFydGlmYWN0IGhhbmRsaW5n'
        'LiIiIgoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgcmUK'
        'ZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgSXRlcmFibGUKCmltcG9ydCBqb2JsaWIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBw'
        'YW5kYXMgYXMgcGQKaW1wb3J0IHNrbGVhcm4KZnJvbSBza2xlYXJuLmNsdXN0ZXIgaW1wb3J0IEtNZWFucwpmcm9tIHNrbGVhcm4u'
        'bWV0cmljcyBpbXBvcnQgc2lsaG91ZXR0ZV9zY29yZQpmcm9tIHNrbGVhcm4ucGlwZWxpbmUgaW1wb3J0IFBpcGVsaW5lCmZyb20g'
        'c2tsZWFybi5wcmVwcm9jZXNzaW5nIGltcG9ydCBTdGFuZGFyZFNjYWxlcgoKCkZFQVRVUkVTID0gKCJyZWNlbmN5IiwgImZyZXF1'
        'ZW5jeSIsICJtb25ldGFyeSIpCgoKY2xhc3MgUkZNRGF0YVByZXBhcmVyOgogICAgIiIiQWdncmVnYXRlIE9ubGluZSBSZXRhaWwg'
        'dHJhbnNhY3Rpb25zIHRvIG9uZSBjdXN0b21lci1sZXZlbCBSRk0gcm93LiIiIgoKICAgIFJFUVVJUkVEX0NPTFVNTlMgPSAoIkN1'
        'c3RvbWVySUQiLCAiSW52b2ljZURhdGUiLCAiSW52b2ljZU5vIiwgIlF1YW50aXR5IiwgIlVuaXRQcmljZSIpCgogICAgZGVmIHBy'
        'ZXBhcmUoc2VsZiwgdHJhbnNhY3Rpb25zOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICBtaXNzaW5nID0g'
        'W25hbWUgZm9yIG5hbWUgaW4gc2VsZi5SRVFVSVJFRF9DT0xVTU5TIGlmIG5hbWUgbm90IGluIHRyYW5zYWN0aW9ucy5jb2x1bW5z'
        'XQogICAgICAgIGlmIG1pc3Npbmc6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJNaXNzaW5nIHJlcXVpcmVkIE9ubGlu'
        'ZSBSZXRhaWwgY29sdW1uczogeycsICcuam9pbihtaXNzaW5nKX0iKQogICAgICAgIGZyYW1lID0gdHJhbnNhY3Rpb25zLmxvY1s6'
        'LCBsaXN0KHNlbGYuUkVRVUlSRURfQ09MVU1OUyldLmNvcHkoKQogICAgICAgIGZyYW1lID0gZnJhbWUuZHJvcG5hKHN1YnNldD1z'
        'ZWxmLlJFUVVJUkVEX0NPTFVNTlMpCiAgICAgICAgZnJhbWVbIkludm9pY2VEYXRlIl0gPSBwZC50b19kYXRldGltZShmcmFtZVsi'
        'SW52b2ljZURhdGUiXSwgZXJyb3JzPSJjb2VyY2UiLCB1dGM9VHJ1ZSkKICAgICAgICBmcmFtZVsiQ3VzdG9tZXJJRCJdID0gcGQu'
        'dG9fbnVtZXJpYyhmcmFtZVsiQ3VzdG9tZXJJRCJdLCBlcnJvcnM9ImNvZXJjZSIpCiAgICAgICAgZnJhbWVbIlF1YW50aXR5Il0g'
        'PSBwZC50b19udW1lcmljKGZyYW1lWyJRdWFudGl0eSJdLCBlcnJvcnM9ImNvZXJjZSIpCiAgICAgICAgZnJhbWVbIlVuaXRQcmlj'
        'ZSJdID0gcGQudG9fbnVtZXJpYyhmcmFtZVsiVW5pdFByaWNlIl0sIGVycm9ycz0iY29lcmNlIikKICAgICAgICBmcmFtZSA9IGZy'
        'YW1lLmRyb3BuYShzdWJzZXQ9WyJJbnZvaWNlRGF0ZSIsICJDdXN0b21lcklEIiwgIlF1YW50aXR5IiwgIlVuaXRQcmljZSJdKQog'
        'ICAgICAgICMgVUNJIFVuaXRQcmljZSBpcyBpbiBHQlA7IGV4Y2x1ZGUgY2FuY2VsbGF0aW9ucyBhbmQgbm9uLXBvc2l0aXZlIHF1'
        'YW50aXRpZXMuCiAgICAgICAgZnJhbWUgPSBmcmFtZS5sb2NbCiAgICAgICAgICAgIH5mcmFtZVsiSW52b2ljZU5vIl0uYXN0eXBl'
        'KHN0cikuc3RyLnVwcGVyKCkuc3RyLnN0YXJ0c3dpdGgoIkMiKQogICAgICAgICAgICAmIChmcmFtZVsiUXVhbnRpdHkiXSA+IDAp'
        'CiAgICAgICAgICAgICYgKGZyYW1lWyJVbml0UHJpY2UiXSA+PSAwKQogICAgICAgIF0KICAgICAgICBpZiBmcmFtZS5lbXB0eToK'
        'ICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTm8gdmFsaWQgdHJhbnNhY3Rpb25zIHJlbWFpbiBhZnRlciBjbGVhbmluZy4i'
        'KQogICAgICAgIGZyYW1lWyJsaW5lX3ZhbHVlIl0gPSBmcmFtZVsiUXVhbnRpdHkiXSAqIGZyYW1lWyJVbml0UHJpY2UiXQogICAg'
        'ICAgIHNuYXBzaG90ID0gZnJhbWVbIkludm9pY2VEYXRlIl0ubWF4KCkubm9ybWFsaXplKCkgKyBwZC5EYXRlT2Zmc2V0KGRheXM9'
        'MSkKICAgICAgICBncm91cGVkID0gZnJhbWUuZ3JvdXBieSgiQ3VzdG9tZXJJRCIsIHNvcnQ9VHJ1ZSkuYWdnKAogICAgICAgICAg'
        'ICBsYXN0X3B1cmNoYXNlPSgiSW52b2ljZURhdGUiLCAibWF4IiksCiAgICAgICAgICAgIGZyZXF1ZW5jeT0oIkludm9pY2VObyIs'
        'ICJudW5pcXVlIiksCiAgICAgICAgICAgIG1vbmV0YXJ5PSgibGluZV92YWx1ZSIsICJzdW0iKSwKICAgICAgICApCiAgICAgICAg'
        'cmVzdWx0ID0gcGQuRGF0YUZyYW1lKGluZGV4PWdyb3VwZWQuaW5kZXgpCiAgICAgICAgcmVzdWx0WyJyZWNlbmN5Il0gPSAoc25h'
        'cHNob3QgLSBncm91cGVkWyJsYXN0X3B1cmNoYXNlIl0uZHQubm9ybWFsaXplKCkpLmR0LmRheXMKICAgICAgICByZXN1bHRbImZy'
        'ZXF1ZW5jeSJdID0gZ3JvdXBlZFsiZnJlcXVlbmN5Il0uYXN0eXBlKGZsb2F0KQogICAgICAgIHJlc3VsdFsibW9uZXRhcnkiXSA9'
        'IGdyb3VwZWRbIm1vbmV0YXJ5Il0uYXN0eXBlKGZsb2F0KQogICAgICAgIHJlc3VsdC5pbmRleC5uYW1lID0gIkN1c3RvbWVySUQi'
        'CiAgICAgICAgaWYgKHJlc3VsdFtsaXN0KEZFQVRVUkVTKV0gPCAwKS5hbnkoKS5hbnkoKSBvciBub3QgbnAuaXNmaW5pdGUocmVz'
        'dWx0W2xpc3QoRkVBVFVSRVMpXSkuYWxsKCkuYWxsKCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlByZXBhcmVkIFJG'
        'TSBmZWF0dXJlcyBtdXN0IGJlIGZpbml0ZSBhbmQgbm9uLW5lZ2F0aXZlLiIpCiAgICAgICAgcmV0dXJuIHJlc3VsdC5sb2NbOiwg'
        'bGlzdChGRUFUVVJFUyldCgoKQGRhdGFjbGFzcwpjbGFzcyBSRk1Nb2RlbEFydGlmYWN0OgogICAgcGlwZWxpbmU6IFBpcGVsaW5l'
        'CiAgICBmZWF0dXJlX25hbWVzOiB0dXBsZVtzdHIsIC4uLl0KICAgIHRyYWluaW5nX3JhbmdlczogZGljdFtzdHIsIHR1cGxlW2Zs'
        'b2F0LCBmbG9hdF1dCiAgICB0cmFpbmluZ19wYXJhbWV0ZXJzOiBkaWN0W3N0ciwgQW55XQogICAgY2x1c3Rlcl9zZWxlY3Rpb246'
        'IGxpc3RbZGljdFtzdHIsIGZsb2F0IHwgaW50IHwgTm9uZV1dCiAgICBzZWdtZW50X21ldGFkYXRhOiBkaWN0W3N0ciwgZGljdFtz'
        'dHIsIEFueV1dCiAgICBkYXRhc2V0X21ldGFkYXRhOiBkaWN0W3N0ciwgQW55XQogICAgc2tsZWFybl92ZXJzaW9uOiBzdHIKCgpj'
        'bGFzcyBSRk1Nb2RlbFRyYWluZXI6CiAgICAiIiJGaXQgYW5kIGV2YWx1YXRlIGEgc2NhbGVyLXBsdXMtSy1NZWFucyBwaXBlbGlu'
        'ZS4iIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBuX2NsdXN0ZXJzOiBpbnQgfCBOb25lID0gTm9u'
        'ZSwKICAgICAgICByYW5kb21fc3RhdGU6IGludCA9IDQyLAogICAgICAgIG5faW5pdDogaW50ID0gMTAsCiAgICApIC0+IE5vbmU6'
        'CiAgICAgICAgaWYgbl9jbHVzdGVycyBpcyBub3QgTm9uZSBhbmQgbl9jbHVzdGVycyA8IDI6CiAgICAgICAgICAgIHJhaXNlIFZh'
        'bHVlRXJyb3IoIm5fY2x1c3RlcnMgbXVzdCBiZSBhdCBsZWFzdCAyLiIpCiAgICAgICAgc2VsZi5uX2NsdXN0ZXJzID0gbl9jbHVz'
        'dGVycwogICAgICAgIHNlbGYucmFuZG9tX3N0YXRlID0gcmFuZG9tX3N0YXRlCiAgICAgICAgc2VsZi5uX2luaXQgPSBuX2luaXQK'
        'ICAgICAgICBzZWxmLnBpcGVsaW5lOiBQaXBlbGluZSB8IE5vbmUgPSBOb25lCiAgICAgICAgc2VsZi5jbHVzdGVyX3NlbGVjdGlv'
        'bjogbGlzdFtkaWN0W3N0ciwgZmxvYXQgfCBpbnQgfCBOb25lXV0gPSBbXQogICAgICAgIHNlbGYuc2VsZWN0ZWRfbl9jbHVzdGVy'
        'czogaW50IHwgTm9uZSA9IE5vbmUKCiAgICBkZWYgZXZhbHVhdGVfY2x1c3Rlcl9jYW5kaWRhdGVzKAogICAgICAgIHNlbGYsCiAg'
        'ICAgICAgcmZtOiBwZC5EYXRhRnJhbWUsCiAgICAgICAgY2FuZGlkYXRlczogSXRlcmFibGVbaW50XSA9IHJhbmdlKDIsIDExKSwK'
        'ICAgICkgLT4gcGQuRGF0YUZyYW1lOgogICAgICAgICIiIkNhbGN1bGF0ZSBpbmVydGlhIGFuZCBzaWxob3VldHRlIGZvciBlYWNo'
        'IHZhbGlkIGNhbmRpZGF0ZSBrLiIiIgogICAgICAgIHNlbGYuX3ZhbGlkYXRlX2ZyYW1lKHJmbSkKICAgICAgICBzY2FsZWQgPSBT'
        'dGFuZGFyZFNjYWxlcigpLmZpdF90cmFuc2Zvcm0ocmZtLmxvY1s6LCBsaXN0KEZFQVRVUkVTKV0pCiAgICAgICAgcm93czogbGlz'
        'dFtkaWN0W3N0ciwgZmxvYXQgfCBpbnQgfCBOb25lXV0gPSBbXQogICAgICAgIGZvciBjYW5kaWRhdGUgaW4gY2FuZGlkYXRlczoK'
        'ICAgICAgICAgICAgaWYgY2FuZGlkYXRlIDwgMiBvciBjYW5kaWRhdGUgPj0gbGVuKHJmbSk6CiAgICAgICAgICAgICAgICBjb250'
        'aW51ZQogICAgICAgICAgICBlc3RpbWF0b3IgPSBLTWVhbnMoCiAgICAgICAgICAgICAgICBuX2NsdXN0ZXJzPWludChjYW5kaWRh'
        'dGUpLAogICAgICAgICAgICAgICAgcmFuZG9tX3N0YXRlPXNlbGYucmFuZG9tX3N0YXRlLAogICAgICAgICAgICAgICAgbl9pbml0'
        'PXNlbGYubl9pbml0LAogICAgICAgICAgICApCiAgICAgICAgICAgIGxhYmVscyA9IGVzdGltYXRvci5maXRfcHJlZGljdChzY2Fs'
        'ZWQpCiAgICAgICAgICAgIHNpbGhvdWV0dGUgPSBOb25lCiAgICAgICAgICAgIGlmIDEgPCBsZW4oc2V0KGxhYmVscykpIDwgbGVu'
        'KGxhYmVscyk6CiAgICAgICAgICAgICAgICBzaWxob3VldHRlID0gZmxvYXQoc2lsaG91ZXR0ZV9zY29yZShzY2FsZWQsIGxhYmVs'
        'cykpCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJuX2NsdXN0'
        'ZXJzIjogaW50KGNhbmRpZGF0ZSksCiAgICAgICAgICAgICAgICAgICAgImluZXJ0aWEiOiBmbG9hdChlc3RpbWF0b3IuaW5lcnRp'
        'YV8pLAogICAgICAgICAgICAgICAgICAgICJzaWxob3VldHRlIjogc2lsaG91ZXR0ZSwKICAgICAgICAgICAgICAgIH0KICAgICAg'
        'ICAgICAgKQogICAgICAgIGlmIG5vdCByb3dzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJBdCBsZWFzdCBvbmUgdmFs'
        'aWQgY2x1c3RlciBjYW5kaWRhdGUgaXMgcmVxdWlyZWQuIikKICAgICAgICBzZWxmLmNsdXN0ZXJfc2VsZWN0aW9uID0gcm93cwog'
        'ICAgICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykKCiAgICBkZWYgc2VsZWN0X25fY2x1c3RlcnMoCiAgICAgICAgc2VsZiwK'
        'ICAgICAgICByZm06IHBkLkRhdGFGcmFtZSwKICAgICAgICBjYW5kaWRhdGVzOiBJdGVyYWJsZVtpbnRdID0gcmFuZ2UoMiwgMTEp'
        'LAogICAgKSAtPiBpbnQ6CiAgICAgICAgIiIiQ2hvb3NlIHRoZSBoaWdoZXN0LXNpbGhvdWV0dGUgY2FuZGlkYXRlLCBicmVha2lu'
        'ZyB0aWVzIGJ5IGxvd2VyIGsuIiIiCiAgICAgICAgc2VsZWN0aW9uID0gc2VsZi5ldmFsdWF0ZV9jbHVzdGVyX2NhbmRpZGF0ZXMo'
        'cmZtLCBjYW5kaWRhdGVzKQogICAgICAgIG1lYXN1cmFibGUgPSBzZWxlY3Rpb24uZHJvcG5hKHN1YnNldD1bInNpbGhvdWV0dGUi'
        'XSkKICAgICAgICBpZiBtZWFzdXJhYmxlLmVtcHR5OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJObyBjYW5kaWRhdGUg'
        'cHJvZHVjZWQgYSB2YWxpZCBzaWxob3VldHRlIHNjb3JlLiIpCiAgICAgICAgYmVzdCA9IG1lYXN1cmFibGUuc29ydF92YWx1ZXMo'
        'CiAgICAgICAgICAgIFsic2lsaG91ZXR0ZSIsICJuX2NsdXN0ZXJzIl0sIGFzY2VuZGluZz1bRmFsc2UsIFRydWVdLCBraW5kPSJz'
        'dGFibGUiCiAgICAgICAgKS5pbG9jWzBdCiAgICAgICAgc2VsZi5uX2NsdXN0ZXJzID0gaW50KGJlc3RbIm5fY2x1c3RlcnMiXSkK'
        'ICAgICAgICBzZWxmLnNlbGVjdGVkX25fY2x1c3RlcnMgPSBzZWxmLm5fY2x1c3RlcnMKICAgICAgICByZXR1cm4gc2VsZi5uX2Ns'
        'dXN0ZXJzCgogICAgZGVmIGZpdChzZWxmLCByZm06IHBkLkRhdGFGcmFtZSkgLT4gUGlwZWxpbmU6CiAgICAgICAgc2VsZi5fdmFs'
        'aWRhdGVfZnJhbWUocmZtKQogICAgICAgIGlmIHNlbGYubl9jbHVzdGVycyBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBSdW50'
        'aW1lRXJyb3IoIkNhbGwgc2VsZWN0X25fY2x1c3RlcnMgYmVmb3JlIGZpdHRpbmcgdGhlIGZpbmFsIG1vZGVsLiIpCiAgICAgICAg'
        'aWYgbGVuKHJmbSkgPCBzZWxmLm5fY2x1c3RlcnM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRyYWluaW5nIHJvd3Mg'
        'bXVzdCBiZSBhdCBsZWFzdCB0aGUgbnVtYmVyIG9mIGNsdXN0ZXJzLiIpCiAgICAgICAgc2VsZi5waXBlbGluZSA9IFBpcGVsaW5l'
        'KAogICAgICAgICAgICBzdGVwcz1bCiAgICAgICAgICAgICAgICAoInNjYWxlciIsIFN0YW5kYXJkU2NhbGVyKCkpLAogICAgICAg'
        'ICAgICAgICAgKCJrbWVhbnMiLCBLTWVhbnMoCiAgICAgICAgICAgICAgICAgICAgbl9jbHVzdGVycz1zZWxmLm5fY2x1c3RlcnMs'
        'CiAgICAgICAgICAgICAgICAgICAgcmFuZG9tX3N0YXRlPXNlbGYucmFuZG9tX3N0YXRlLAogICAgICAgICAgICAgICAgICAgIG5f'
        'aW5pdD1zZWxmLm5faW5pdCwKICAgICAgICAgICAgICAgICkpLAogICAgICAgICAgICBdCiAgICAgICAgKQogICAgICAgIHNlbGYu'
        'cGlwZWxpbmUuZml0KHJmbS5sb2NbOiwgbGlzdChGRUFUVVJFUyldKQogICAgICAgIHJldHVybiBzZWxmLnBpcGVsaW5lCgogICAg'
        'ZGVmIGV2YWx1YXRlKHNlbGYsIHJmbTogcGQuRGF0YUZyYW1lKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICAgICBpZiBzZWxmLnBp'
        'cGVsaW5lIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQ2FsbCBmaXQgYmVmb3JlIGV2YWx1YXRlLiIp'
        'CiAgICAgICAgbGFiZWxzID0gc2VsZi5waXBlbGluZS5wcmVkaWN0KHJmbS5sb2NbOiwgbGlzdChGRUFUVVJFUyldKQogICAgICAg'
        'IG1vZGVsID0gc2VsZi5waXBlbGluZS5uYW1lZF9zdGVwc1sia21lYW5zIl0KICAgICAgICByZXN1bHQ6IGRpY3Rbc3RyLCBBbnld'
        'ID0gewogICAgICAgICAgICAiaW5lcnRpYSI6IGZsb2F0KG1vZGVsLmluZXJ0aWFfKSwKICAgICAgICAgICAgIm5fY2x1c3RlcnMi'
        'OiBpbnQobW9kZWwubl9jbHVzdGVycyksCiAgICAgICAgICAgICJjbHVzdGVyX2NvdW50cyI6IHBkLlNlcmllcyhsYWJlbHMpLnZh'
        'bHVlX2NvdW50cygpLnNvcnRfaW5kZXgoKS50b19kaWN0KCksCiAgICAgICAgfQogICAgICAgIGlmIGxlbihzZXQobGFiZWxzKSkg'
        'PiAxIGFuZCBsZW4oc2V0KGxhYmVscykpIDwgbGVuKGxhYmVscyk6CiAgICAgICAgICAgIHNjYWxlZCA9IHNlbGYucGlwZWxpbmUu'
        'bmFtZWRfc3RlcHNbInNjYWxlciJdLnRyYW5zZm9ybShyZm0ubG9jWzosIGxpc3QoRkVBVFVSRVMpXSkKICAgICAgICAgICAgcmVz'
        'dWx0WyJzaWxob3VldHRlIl0gPSBmbG9hdChzaWxob3VldHRlX3Njb3JlKHNjYWxlZCwgbGFiZWxzKSkKICAgICAgICBlbHNlOgog'
        'ICAgICAgICAgICByZXN1bHRbInNpbGhvdWV0dGUiXSA9IE5vbmUKICAgICAgICByZXR1cm4gcmVzdWx0CgogICAgZGVmIGNyZWF0'
        'ZV9hcnRpZmFjdCgKICAgICAgICBzZWxmLAogICAgICAgIHJmbTogcGQuRGF0YUZyYW1lLAogICAgICAgIGRhdGFzZXRfbWV0YWRh'
        'dGE6IGRpY3Rbc3RyLCBBbnldIHwgTm9uZSA9IE5vbmUsCiAgICApIC0+IFJGTU1vZGVsQXJ0aWZhY3Q6CiAgICAgICAgaWYgc2Vs'
        'Zi5waXBlbGluZSBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNhbGwgZml0IGJlZm9yZSBjcmVhdGlu'
        'ZyBhbiBhcnRpZmFjdC4iKQogICAgICAgIGlmIHNlbGYuc2VsZWN0ZWRfbl9jbHVzdGVycyAhPSBzZWxmLm5fY2x1c3RlcnMgb3Ig'
        'bm90IHNlbGYuY2x1c3Rlcl9zZWxlY3Rpb246CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiU2VsZWN0IHRoZSBjbHVz'
        'dGVyIGNvdW50IGJlZm9yZSBjcmVhdGluZyB0aGUgZmluYWwgYXJ0aWZhY3QuIikKICAgICAgICBzZWxmLl92YWxpZGF0ZV9mcmFt'
        'ZShyZm0pCiAgICAgICAgcmFuZ2VzID0gewogICAgICAgICAgICBmZWF0dXJlOiAoZmxvYXQocmZtW2ZlYXR1cmVdLm1pbigpKSwg'
        'ZmxvYXQocmZtW2ZlYXR1cmVdLm1heCgpKSkKICAgICAgICAgICAgZm9yIGZlYXR1cmUgaW4gRkVBVFVSRVMKICAgICAgICB9CiAg'
        'ICAgICAgbW9kZWwgPSBzZWxmLnBpcGVsaW5lLm5hbWVkX3N0ZXBzWyJrbWVhbnMiXQogICAgICAgIGNlbnRlcnMgPSBzZWxmLnBp'
        'cGVsaW5lLm5hbWVkX3N0ZXBzWyJzY2FsZXIiXS5pbnZlcnNlX3RyYW5zZm9ybShtb2RlbC5jbHVzdGVyX2NlbnRlcnNfKQogICAg'
        'ICAgIGxhYmVscyA9IHNlbGYucGlwZWxpbmUucHJlZGljdChyZm0ubG9jWzosIGxpc3QoRkVBVFVSRVMpXSkKICAgICAgICBzZWdt'
        'ZW50X21ldGFkYXRhOiBkaWN0W3N0ciwgZGljdFtzdHIsIEFueV1dID0ge30KICAgICAgICBmb3IgbGFiZWwgaW4gcmFuZ2UobW9k'
        'ZWwubl9jbHVzdGVycyk6CiAgICAgICAgICAgIHNlZ21lbnRfbWV0YWRhdGFbc3RyKGxhYmVsKV0gPSB7CiAgICAgICAgICAgICAg'
        'ICAic2VnbWVudF9pZCI6IGludChsYWJlbCksCiAgICAgICAgICAgICAgICAiY3VzdG9tZXJfY291bnQiOiBpbnQobnAuY291bnRf'
        'bm9uemVybyhsYWJlbHMgPT0gbGFiZWwpKSwKICAgICAgICAgICAgICAgICJjZW50cm9pZCI6IHsKICAgICAgICAgICAgICAgICAg'
        'ICBmZWF0dXJlOiBmbG9hdChjZW50ZXJzW2xhYmVsLCBjb2x1bW5faW5kZXhdKQogICAgICAgICAgICAgICAgICAgIGZvciBjb2x1'
        'bW5faW5kZXgsIGZlYXR1cmUgaW4gZW51bWVyYXRlKEZFQVRVUkVTKQogICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgfQog'
        'ICAgICAgIHJldHVybiBSRk1Nb2RlbEFydGlmYWN0KAogICAgICAgICAgICBwaXBlbGluZT1zZWxmLnBpcGVsaW5lLAogICAgICAg'
        'ICAgICBmZWF0dXJlX25hbWVzPUZFQVRVUkVTLAogICAgICAgICAgICB0cmFpbmluZ19yYW5nZXM9cmFuZ2VzLAogICAgICAgICAg'
        'ICB0cmFpbmluZ19wYXJhbWV0ZXJzPXsKICAgICAgICAgICAgICAgICJuX2NsdXN0ZXJzIjogc2VsZi5uX2NsdXN0ZXJzLAogICAg'
        'ICAgICAgICAgICAgInJhbmRvbV9zdGF0ZSI6IHNlbGYucmFuZG9tX3N0YXRlLAogICAgICAgICAgICAgICAgIm5faW5pdCI6IHNl'
        'bGYubl9pbml0LAogICAgICAgICAgICAgICAgInNlbGVjdGlvbl9jcml0ZXJpb24iOiAibWF4aW11bV9zaWxob3VldHRlX3RoZW5f'
        'c21hbGxlc3RfayIsCiAgICAgICAgICAgIH0sCiAgICAgICAgICAgIGNsdXN0ZXJfc2VsZWN0aW9uPWxpc3Qoc2VsZi5jbHVzdGVy'
        'X3NlbGVjdGlvbiksCiAgICAgICAgICAgIHNlZ21lbnRfbWV0YWRhdGE9c2VnbWVudF9tZXRhZGF0YSwKICAgICAgICAgICAgZGF0'
        'YXNldF9tZXRhZGF0YT1kaWN0KGRhdGFzZXRfbWV0YWRhdGEgb3IgeyJkYXRhc2V0X25hbWUiOiAidW5zcGVjaWZpZWQifSksCiAg'
        'ICAgICAgICAgIHNrbGVhcm5fdmVyc2lvbj1za2xlYXJuLl9fdmVyc2lvbl9fLAogICAgICAgICkKCiAgICBAc3RhdGljbWV0aG9k'
        'CiAgICBkZWYgc2F2ZV9hcnRpZmFjdCgKICAgICAgICBhcnRpZmFjdDogUkZNTW9kZWxBcnRpZmFjdCwKICAgICAgICBwYXRoOiBQ'
        'YXRoLAogICAgICAgIGFsbG93X3Rlc3RfYXJ0aWZhY3Q6IGJvb2wgPSBGYWxzZSwKICAgICkgLT4gTm9uZToKICAgICAgICBpZiBu'
        'b3QgYWxsb3dfdGVzdF9hcnRpZmFjdDoKICAgICAgICAgICAgUkZNTW9kZWxUcmFpbmVyLl92YWxpZGF0ZV9vZmZpY2lhbF9kYXRh'
        'c2V0X21ldGFkYXRhKGFydGlmYWN0LmRhdGFzZXRfbWV0YWRhdGEpCiAgICAgICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1U'
        'cnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIGpvYmxpYi5kdW1wKGFydGlmYWN0LCBwYXRoKQoKICAgIEBzdGF0aWNtZXRob2QK'
        'ICAgIGRlZiBsb2FkX2FydGlmYWN0KHBhdGg6IFBhdGgsIGFsbG93X3Rlc3RfYXJ0aWZhY3Q6IGJvb2wgPSBGYWxzZSkgLT4gUkZN'
        'TW9kZWxBcnRpZmFjdDoKICAgICAgICBhcnRpZmFjdCA9IGpvYmxpYi5sb2FkKHBhdGgpCiAgICAgICAgaWYgbm90IGlzaW5zdGFu'
        'Y2UoYXJ0aWZhY3QsIFJGTU1vZGVsQXJ0aWZhY3QpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJNb2RlbCBmaWxlIGRv'
        'ZXMgbm90IGNvbnRhaW4gYSBzdXBwb3J0ZWQgUkZNTW9kZWxBcnRpZmFjdC4iKQogICAgICAgIGlmIGFydGlmYWN0LmZlYXR1cmVf'
        'bmFtZXMgIT0gRkVBVFVSRVM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1vZGVsIGFydGlmYWN0IGhhcyBhbiBpbmNv'
        'bXBhdGlibGUgUkZNIGZlYXR1cmUgb3JkZXIuIikKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShhcnRpZmFjdC5waXBlbGluZSwg'
        'UGlwZWxpbmUpIG9yIG5vdCB7InNjYWxlciIsICJrbWVhbnMifS5pc3N1YnNldChhcnRpZmFjdC5waXBlbGluZS5uYW1lZF9zdGVw'
        'cyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1vZGVsIGFydGlmYWN0IG11c3QgaW5jbHVkZSBzY2FsZXIgYW5kIEst'
        'TWVhbnMgc3RlcHMuIikKICAgICAgICBpZiBhcnRpZmFjdC5za2xlYXJuX3ZlcnNpb24uc3BsaXQoIi4iKVs6Ml0gIT0gc2tsZWFy'
        'bi5fX3ZlcnNpb25fXy5zcGxpdCgiLiIpWzoyXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAg'
        'ICJNb2RlbCBhcnRpZmFjdCBzY2lraXQtbGVhcm4gdmVyc2lvbiBpcyBpbmNvbXBhdGlibGUgd2l0aCB0aGUgcnVudGltZSAiCiAg'
        'ICAgICAgICAgICAgICBmIih7YXJ0aWZhY3Quc2tsZWFybl92ZXJzaW9ufSB2cyB7c2tsZWFybi5fX3ZlcnNpb25fX30pLiIKICAg'
        'ICAgICAgICAgKQogICAgICAgIGlmIHNldChhcnRpZmFjdC50cmFpbmluZ19yYW5nZXMpICE9IHNldChGRUFUVVJFUyk6CiAgICAg'
        'ICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1vZGVsIGFydGlmYWN0IGlzIG1pc3NpbmcgdHJhaW5pbmcgcmFuZ2VzIGZvciBSRk0g'
        'ZmVhdHVyZXMuIikKICAgICAgICBpZiBub3QgYXJ0aWZhY3Quc2VnbWVudF9tZXRhZGF0YSBvciBzZXQoYXJ0aWZhY3Quc2VnbWVu'
        'dF9tZXRhZGF0YSkgIT0gewogICAgICAgICAgICBzdHIoaW5kZXgpIGZvciBpbmRleCBpbiByYW5nZShhcnRpZmFjdC50cmFpbmlu'
        'Z19wYXJhbWV0ZXJzWyJuX2NsdXN0ZXJzIl0pCiAgICAgICAgfToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTW9kZWwg'
        'YXJ0aWZhY3QgaXMgbWlzc2luZyBwZXItc2VnbWVudCBtZXRhZGF0YS4iKQogICAgICAgIGlmIG5vdCBhbGxvd190ZXN0X2FydGlm'
        'YWN0OgogICAgICAgICAgICBSRk1Nb2RlbFRyYWluZXIuX3ZhbGlkYXRlX29mZmljaWFsX2RhdGFzZXRfbWV0YWRhdGEoYXJ0aWZh'
        'Y3QuZGF0YXNldF9tZXRhZGF0YSkKICAgICAgICByZXR1cm4gYXJ0aWZhY3QKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX3Zh'
        'bGlkYXRlX29mZmljaWFsX2RhdGFzZXRfbWV0YWRhdGEobWV0YWRhdGE6IGRpY3Rbc3RyLCBBbnldKSAtPiBOb25lOgogICAgICAg'
        'IGRpZ2VzdCA9IHN0cihtZXRhZGF0YS5nZXQoInNoYTI1NiIsICIiKSkKICAgICAgICBpZiAoCiAgICAgICAgICAgIG1ldGFkYXRh'
        'LmdldCgiZGF0YXNldF9pZCIpICE9IDM1MgogICAgICAgICAgICBvciBtZXRhZGF0YS5nZXQoImRhdGFzZXRfbmFtZSIpICE9ICJP'
        'bmxpbmUgUmV0YWlsIgogICAgICAgICAgICBvciBtZXRhZGF0YS5nZXQoInRyYW5zYWN0aW9uX2NvdW50IikgIT0gNTQxXzkwOQog'
        'ICAgICAgICAgICBvciBtZXRhZGF0YS5nZXQoInNvdXJjZSIpIG5vdCBpbiB7InVjaS1vZmZpY2lhbC1kb3dubG9hZCIsICJsb2Nh'
        'bC11Y2ktd29ya2Jvb2sifQogICAgICAgICAgICBvciByZS5mdWxsbWF0Y2gociJbMC05YS1mXXs2NH0iLCBkaWdlc3QpIGlzIE5v'
        'bmUKICAgICAgICApOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgIkZpbmFsIG1vZGVsIGFy'
        'dGlmYWN0cyBtdXN0IHJlY29yZCBVQ0kgT25saW5lIFJldGFpbCBkYXRhc2V0IDM1MiwgIgogICAgICAgICAgICAgICAgIjU0MSw5'
        'MDkgdHJhbnNhY3Rpb25zLCBhbiBhY2NlcHRlZCBVQ0kgYWNxdWlzaXRpb24gbWV0aG9kLCBhbmQgdGhlIHdvcmtib29rIFNIQS0y'
        'NTYuIgogICAgICAgICAgICApCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF92YWxpZGF0ZV9mcmFtZShyZm06IHBkLkRhdGFG'
        'cmFtZSkgLT4gTm9uZToKICAgICAgICBpZiB0dXBsZShyZm0uY29sdW1ucykgIT0gRkVBVFVSRVM6CiAgICAgICAgICAgIHJhaXNl'
        'IFZhbHVlRXJyb3IoZiJSRk0gY29sdW1ucyBtdXN0IGJlIG9yZGVyZWQgYXMge0ZFQVRVUkVTfS4iKQogICAgICAgIGlmIHJmbS5l'
        'bXB0eSBvciBub3QgbnAuaXNmaW5pdGUocmZtLnRvX251bXB5KGR0eXBlPWZsb2F0KSkuYWxsKCk6CiAgICAgICAgICAgIHJhaXNl'
        'IFZhbHVlRXJyb3IoIlJGTSBkYXRhIG11c3QgYmUgbm9uLWVtcHR5IGFuZCBmaW5pdGUuIikKICAgICAgICBpZiAocmZtIDwgMCku'
        'YW55KCkuYW55KCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlJGTSB2YWx1ZXMgbXVzdCBiZSBub24tbmVnYXRpdmUu'
        'IikK'
    ),
}
package_parent = Path(tempfile.gettempdir()) / 'segmenta_rfm_atlas_notebook_package'
package_dir = package_parent / 'segmenta_rfm_atlas'
package_dir.mkdir(parents=True, exist_ok=True)
for module_name, encoded_source in embedded_sources.items():
    (package_dir / module_name).write_bytes(base64.b64decode(encoded_source))
if str(package_parent) not in sys.path:
    sys.path.insert(0, str(package_parent))
print(f'Notebook training package ready: {package_dir}')

Notebook training package ready: /tmp/segmenta_rfm_atlas_notebook_package/segmenta_rfm_atlas


In [2]:
import os
from pathlib import Path
import pandas as pd
from segmenta_rfm_atlas.dataset import DatasetLoadError, OnlineRetailDatasetLoader
from segmenta_rfm_atlas.modeling import RFMDataPreparer, RFMModelTrainer

## Load the official UCI workbook

The loader downloads the official UCI dataset 352 ZIP when the local workbook is missing. In Colab, a failed download opens the file picker for the same official `Online Retail.xlsx`. A file with any transaction count other than 541,909 is rejected; no training or artifact export occurs. Locally, place the official workbook at `data/Online Retail.xlsx` or set `ONLINE_RETAIL_PATH`.

In [3]:
data_path = Path(os.getenv('ONLINE_RETAIL_PATH', 'data/Online Retail.xlsx'))
dataset_loader = OnlineRetailDatasetLoader()
try:
    loaded_dataset = dataset_loader.load(data_path, auto_download=True)
except DatasetLoadError as download_error:
    print(f'UCI automatic load failed: {download_error}')
    if dataset_loader.is_google_colab():
        dataset_loader.upload_manual_workbook(data_path)
        loaded_dataset = dataset_loader.load(data_path, auto_download=False)
    else:
        raise FileNotFoundError(
            'Download failed. Place the official 541,909-row Online Retail.xlsx at '
            f'{data_path} and rerun this cell.'
        ) from download_error
transactions = loaded_dataset.transactions
dataset_metadata = loaded_dataset.metadata
assert len(transactions) == OnlineRetailDatasetLoader.EXPECTED_TRANSACTION_COUNT
dataset_metadata

{'dataset_id': 352,
 'dataset_name': 'Online Retail',
 'source': 'uci-official-download',
 'source_url': 'https://archive.ics.uci.edu/static/public/352/online+retail.zip',
 'transaction_count': 541909,
 'sha256': '43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d'}

## Prepare customer-level RFM

The preparer drops unusable customer/transaction fields, excludes cancelled invoices (invoice numbers beginning with `C`) and non-positive quantities, and calculates spend as `Quantity * UnitPrice` in the dataset's sterling units. Frequency counts distinct invoice numbers. Recency is measured in whole days against the day after the latest valid invoice date. The raw-source row count remains recorded separately from the customer-level training row count.

In [4]:
preparer = RFMDataPreparer()
rfm = preparer.prepare(transactions)
rfm.head()

,recency,frequency,monetary
CustomerID,,,
12346.0,326,1.0,77183.60
12347.0,3,7.0,4310.00
12348.0,76,4.0,1797.24
12349.0,19,1.0,1757.55
12350.0,311,1.0,334.40


## Select the number of clusters and evaluate

For each valid candidate `k` from 2 through 10, the trainer calculates inertia and silhouette on the RFM customer observations. It selects the highest silhouette score, breaking ties with the smaller k, then fits the final `StandardScaler` + `KMeans` pipeline using that selected k. The table and final model metrics below are calculated from the loaded UCI data at execution time.

In [5]:
trainer = RFMModelTrainer(random_state=42, n_init=10)
selected_n_clusters = trainer.select_n_clusters(rfm, candidates=range(2, 11))
cluster_selection = pd.DataFrame(trainer.cluster_selection)
print(f'Selected k by maximum silhouette (tie: smaller k): {selected_n_clusters}')
display(cluster_selection)
pipeline = trainer.fit(rfm)
metrics = trainer.evaluate(rfm)
metrics

Selected k by maximum silhouette (tie: smaller k): 2


,n_clusters,inertia,silhouette
0,2,9014.184372,0.895853
1,3,5440.711220,0.593871
2,4,4093.744251,0.616098
3,5,3121.071108,0.615749
4,6,2476.883001,0.587777
5,7,2022.315907,0.521647
6,8,1716.007503,0.485843
7,9,1446.079502,0.477835
8,10,1283.267271,0.477995


{'inertia': 9014.184371553807,
 'n_clusters': 2,
 'cluster_counts': {0: 4313, 1: 26},
 'silhouette': 0.8958528715697585}

## Save the final reusable artifact

The artifact contains the fitted scaler and K-Means pipeline, ordered features, training ranges, selected-k metrics/parameters, UCI dataset provenance and SHA-256, plus each segment's ID, customer count, and centroid in original RFM units. The Flask app loads this artifact at startup and does not retrain. Generated model files and the dataset are excluded from Git.

In [6]:
artifact_path = Path(os.getenv('MODEL_PATH', 'artifacts/rfm_kmeans.joblib'))
artifact = trainer.create_artifact(rfm, dataset_metadata=dataset_metadata)
trainer.save_artifact(artifact, artifact_path)
print(f'Final UCI model artifact saved to {artifact_path}')

Final UCI model artifact saved to artifacts/rfm_kmeans.joblib
